# BDLegalRAG — QA Pair Generation (Stage 2.5: Evaluation Set Preparation)

এই notebook-এ আমরা Stage 1-2-এর corpus থেকে **Retrieval evaluation-এর জন্য গোল্ড QA set** বানাব।

**পদ্ধতি (semi-automated, paper-এ transparent ভাবে লেখার মতো):**
1. Corpus থেকে ~150-200টা Section proportionally sample করা
2. প্রতিটার জন্য Gemini API দিয়ে draft প্রশ্ন+উত্তর তৈরি করা
3. CSV-তে সেভ করা — যাতে আপনি নিজে/আপনার সাথে থাকা কেউ manually verify/correct করতে পারেন
4. Verified rows-ই হবে আপনার final gold QA set

**গুরুত্বপূর্ণ:** ধাপ ২-এর জন্য আপনার নিজের **Gemini API key** লাগবে (নিচে বসাতে হবে)। বাকি সব ধাপ key ছাড়াই চলবে।


## ধাপ ০: Setup + আগের Corpus Load করুন


In [ ]:
import sys, os, json, random, re

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/BDLegalRAG"
else:
    BASE_DIR = "/kaggle/working/BDLegalRAG"

CORPUS_PATH = os.path.join(BASE_DIR, "processed_data", "bdlegalrag_corpus_v1.json")
QA_DIR = os.path.join(BASE_DIR, "qa_data")
os.makedirs(QA_DIR, exist_ok=True)

with open(CORPUS_PATH, encoding="utf-8") as f:
    corpus = json.load(f)

print("Corpus meta:", corpus["meta"])
print("Total acts:", len(corpus["acts"]))


Mounted at /content/drive
Corpus meta: {'source': 'BLAD (Bangladesh Legal Acts Dataset)', 'source_citation': 'Sakhawat, A. (2025). BLAD: A Historically Contextualized, Multilingual Dataset of Bangladeshi Legal Acts (1799 to 2025). arXiv:2607.17111.', 'excluded_repealed': True, 'focused_subset': True, 'domain_keywords_bn': ['অপরাধ', 'দণ্ডবিধি', 'ফৌজদারি', 'বিবাহ', 'তালাক', 'যৌতুক', 'নারী', 'শিশু', 'পারিবারিক', 'নির্যাতন', 'শ্রম', 'শ্রমিক', 'সম্পত্তি', 'ভূমি', 'জমি', 'উত্তরাধিকার'], 'language_filtered': True, 'target_language': 'bengali', 'post_1971_only': False, 'total_acts': 49, 'total_sections': 1575, 'total_tokens': 183635, 'year_range': [1985, 2025], 'language_distribution': {'bn': 1561, 'mixed': 13, 'en': 1}}
Total acts: 49


In [ ]:
!pip install google-generativeai pandas --quiet
import pandas as pd
print("Libraries ready.")


Libraries ready.


## ধাপ ১: Proportional Sampling — কোন Section থেকে প্রশ্ন বানাবেন

সব section থেকে প্রশ্ন বানানো দরকার নেই। **প্রতিটা Act থেকে অন্তত ১টা, এবং বড় Act থেকে বেশি** — এভাবে proportionally sample করে মোট ~TARGET_QA_COUNT টা section বাছাই করা হচ্ছে।

শুধু খুব ছোট (< 30 শব্দ) section বাদ দেওয়া হচ্ছে, কারণ সেগুলো থেকে অর্থবহ প্রশ্ন বানানো কঠিন।


In [ ]:
TARGET_QA_COUNT = 180  # আপনার সিদ্ধান্ত অনুযায়ী বদলান (১৫০-২০০ রেঞ্জে রাখা ভালো)
MIN_SECTION_WORDS = 30
random.seed(42)  # reproducibility-র জন্য fixed seed

# সব section flatten করে metadata সহ একটা লিস্টে আনা
all_sections = []
for act in corpus["acts"]:
    for sec in act["sections"]:
        word_count = len(sec["section_text"].split())
        if word_count < MIN_SECTION_WORDS:
            continue
        all_sections.append({
            "act_title": act["act_title"],
            "act_no": act.get("act_no"),
            "act_year": act.get("act_year"),
            "section_number": sec.get("section_number"),
            "section_text": sec["section_text"],
        })

print(f"Eligible sections (>= {MIN_SECTION_WORDS} words): {len(all_sections)}")

# Act অনুযায়ী গ্রুপ করে, প্রতিটা Act থেকে অন্তত ১টা, বাকিটা random থেকে proportional sample
from collections import defaultdict
by_act = defaultdict(list)
for s in all_sections:
    by_act[s["act_title"]].append(s)

sampled = []
# প্রথমে প্রতিটা Act থেকে ১টা করে (coverage নিশ্চিত করতে)
for act_title, secs in by_act.items():
    sampled.append(random.choice(secs))

# বাকি quota randomly পূরণ করা (যেগুলো এখনো নেওয়া হয়নি তার থেকে)
remaining_needed = max(0, TARGET_QA_COUNT - len(sampled))
already_picked_texts = set(s["section_text"] for s in sampled)
pool = [s for s in all_sections if s["section_text"] not in already_picked_texts]
random.shuffle(pool)
sampled.extend(pool[:remaining_needed])

sampled = sampled[:TARGET_QA_COUNT]
print(f"Total sampled sections for QA generation: {len(sampled)}")


Eligible sections (>= 30 words): 1290
Total sampled sections for QA generation: 180


## ধাপ ২: Gemini API দিয়ে Draft প্রশ্ন-উত্তর তৈরি করা

**এখানে আপনার নিজের Gemini API key বসান** (Google AI Studio থেকে ফ্রি পাওয়া যায়: https://aistudio.google.com/apikey)।

প্রতিটা section-এর জন্য prompt-এ স্পষ্ট করে বলা হচ্ছে: প্রশ্নটা যেন সাধারণ মানুষ যেভাবে জিজ্ঞেস করবে সেভাবে হয় (আইনি জার্গন এড়িয়ে), আর উত্তরটা শুধু দেওয়া text থেকেই আসুক (hallucination এড়াতে)।


In [ ]:
# NOTE (Sept 2026): gemini-3.6-flash's free tier is only 20 requests/day — too
# low for bulk generation. Switching to gemini-2.5-flash-lite, an established
# model with a much higher free daily quota, and adding automatic retry-on-429
# with backoff so transient rate limits don't silently fail into the CSV.

!pip install -U google-genai --quiet

import re, time
from google import genai
from google.genai import types

import os
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")  # <-- এখানে আপনার নিজের Gemini API key বসান
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_NAME = "gemini-2.5-flash-lite"  # higher free-tier daily quota than 3.6-flash
                                        # check https://ai.google.dev/gemini-api/docs/models if this changes

QA_GENERATION_PROMPT = """তুমি একজন বাংলাদেশী আইনি বিশেষজ্ঞ। নিচের আইনের section পড়ে:

১. একটা স্বাভাবিক বাংলা প্রশ্ন বানাও, যেভাবে একজন সাধারণ মানুষ (আইনজীবী নন) এই বিষয়ে জিজ্ঞেস করতে পারে। আইনি জার্গন এড়িয়ে চলো।
২. সেই প্রশ্নের উত্তর দাও, শুধুমাত্র নিচের text-এ যা আছে তার ভিত্তিতে — নিজে থেকে কিছু যোগ কোরো না।

Act: {act_title}
Section: {section_number}
Text: {section_text}

উত্তর এই format-এ দাও, শুধু এই দুই লাইন, আর কিছু না:
প্রশ্ন: <তোমার তৈরি প্রশ্ন>
উত্তর: <তোমার তৈরি উত্তর>
"""

MAX_RETRIES = 3

def generate_qa_draft(section):
    prompt = QA_GENERATION_PROMPT.format(
        act_title=section["act_title"],
        section_number=section.get("section_number") or "N/A",
        section_text=section["section_text"][:1500],
    )
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3),
            )
            text = response.text.strip()
            q_match = re.search(r"প্রশ্ন[:\s]*(.+)", text)
            a_match = re.search(r"উত্তর[:\s]*(.+)", text, re.DOTALL)
            question = q_match.group(1).strip() if q_match else None
            answer = a_match.group(1).strip() if a_match else None
            return question, answer
        except Exception as e:
            msg = str(e)
            if "RESOURCE_EXHAUSTED" in msg or "429" in msg:
                # try to read the retryDelay Google suggests; otherwise back off progressively
                delay_match = re.search(r"retryDelay['\"]?:\s*['\"]?(\d+)", msg)
                wait_s = int(delay_match.group(1)) + 2 if delay_match else (10 * (attempt + 1))
                print(f"  Rate limited, waiting {wait_s}s (attempt {attempt+1}/{MAX_RETRIES})...")
                time.sleep(wait_s)
                continue
            return None, f"ERROR: {e}"
    return None, "ERROR: exhausted retries — daily quota likely used up, stop and resume tomorrow"

print(f"Generation function ready ({MODEL_NAME}, with retry-on-429).")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.
Generation function ready (gemini-2.5-flash-lite, with retry-on-429).


In [ ]:
# Sanity check: confirm MODEL_NAME actually exists for your API key right now.
# Model names change frequently — if generate_content 404s again later, run this
# cell to see what your key currently has access to, and update MODEL_NAME above.
try:
    available = [m.name for m in client.models.list()]
    matching = [m for m in available if "flash" in m.lower()]
    print("Available flash models for your key:")
    for m in matching:
        print(" -", m)
    if not any(MODEL_NAME in m for m in available):
        print(f"\nWARNING: '{MODEL_NAME}' not found in your available models — update MODEL_NAME above.")
except Exception as e:
    print("Could not list models:", e)


Available flash models for your key:
 - models/gemini-2.5-flash
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/gemini-3.1-flash-tts-preview
 - models/gemini-2.5-flash-native-audio-latest
 - models/gemini-2.5-flash-native-audio-preview-09-2025
 - models/gemini-2.5-flash-native-audio-preview-12-2025
 - models/gemini-3.1-flash-live-preview


In [ ]:
import time

qa_rows = []
CHECKPOINT_EVERY = 20
qa_csv_path = os.path.join(QA_DIR, "qa_pairs_draft.csv")

# আগের progress থাকলে resume করা
start_idx = 0
if os.path.exists(qa_csv_path):
    existing_df = pd.read_csv(qa_csv_path)
    qa_rows = existing_df.to_dict("records")
    start_idx = len(qa_rows)
    print(f"Resuming from {start_idx} already-generated pairs.")

for i in range(start_idx, len(sampled)):
    section = sampled[i]
    question, answer = generate_qa_draft(section)

    qa_rows.append({
        "qa_id": f"qa_{i+1:04d}",
        "question": question,
        "model_draft_answer": answer,
        "act_title": section["act_title"],
        "act_no": section["act_no"],
        "act_year": section["act_year"],
        "section_number": section["section_number"],
        "source_text": section["section_text"],
        "human_verified": "",       # আপনি পূরণ করবেন: TRUE/FALSE
        "corrected_answer": "",     # ভুল থাকলে এখানে ঠিক করা উত্তর লিখবেন
        "notes": "",
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(qa_rows).to_csv(qa_csv_path, index=False, encoding="utf-8-sig")
        print(f"[{i+1}/{len(sampled)}] checkpoint saved")

    time.sleep(1.2)  # free-tier rate limit-এর জন্য respectful delay

pd.DataFrame(qa_rows).to_csv(qa_csv_path, index=False, encoding="utf-8-sig")
print(f"Done. {len(qa_rows)} draft QA pairs saved to {qa_csv_path}")


## [PATCH] section_number ঠিক করুন — নতুন API call ছাড়াই

**এই cell-টা তখনই চালান যদি Stage 1-2 notebook-এ section-parsing regex ঠিক করে corpus আবার regenerate করে থাকেন, এবং QA generation ইতিমধ্যে সম্পূর্ণ হয়ে গেছে।**

এটা Gemini API-কে আবার call করে না (quota বাঁচবে) — শুধু `source_text` মিলিয়ে পুরনো CSV-তে সঠিক `section_number` বসিয়ে দেয়, কারণ প্রশ্ন-উত্তর আগেই ঠিকভাবে তৈরি হয়ে আছে, শুধু metadata ভুল ছিল।


In [ ]:
import json, os
import pandas as pd

# নতুন (regex-fixed) corpus থেকে source_text -> section_number lookup বানানো
CORPUS_PATH = os.path.join(BASE_DIR, "processed_data", "bdlegalrag_corpus_v1.json")
with open(CORPUS_PATH, encoding="utf-8") as f:
    fixed_corpus = json.load(f)

text_to_section_number = {}
for act in fixed_corpus["acts"]:
    for sec in act["sections"]:
        text_to_section_number[sec["section_text"]] = sec.get("section_number")

print(f"Lookup built from {len(text_to_section_number)} sections in the corrected corpus.")

# পুরনো QA CSV লোড করা
qa_csv_path = os.path.join(QA_DIR, "qa_pairs_draft.csv")
df = pd.read_csv(qa_csv_path)

before_missing = df["section_number"].isna().sum()

def lookup_section_number(row):
    # যদি আগেই section_number থাকে ঠিকঠাক, সেটাই রাখা; না থাকলে lookup থেকে বসানো
    if pd.notna(row["section_number"]):
        return row["section_number"]
    return text_to_section_number.get(row["source_text"], None)

df["section_number"] = df.apply(lookup_section_number, axis=1)

after_missing = df["section_number"].isna().sum()
print(f"Missing section_number: {before_missing} -> {after_missing}")

df.to_csv(qa_csv_path, index=False, encoding="utf-8-sig")
print(f"Patched CSV saved to {qa_csv_path}")

if after_missing > 0:
    print(f"\nNOTE: {after_missing} rows still missing a section_number — "
          f"these likely need manual inspection (source_text may not have matched exactly).")


Lookup built from 1571 sections in the corrected corpus.
Missing section_number: 0 -> 0
Patched CSV saved to /content/drive/MyDrive/BDLegalRAG/qa_data/qa_pairs_draft.csv


### Missing `section_number` Rows for Manual Inspection

The following rows still have missing `section_number`. Please inspect the `source_text` for these entries and manually update the `section_number` in the `qa_pairs_draft.csv` file, or consider updating them programmatically if you can identify a pattern.

In [ ]:
import pandas as pd
import os

# Reload the DataFrame to ensure we have the latest state
qa_csv_path = os.path.join(QA_DIR, "qa_pairs_draft.csv")
df = pd.read_csv(qa_csv_path)

# Filter and display rows with missing section_number
missing_section_df = df[df["section_number"].isna()]
display(missing_section_df[['qa_id', 'act_title', 'section_number', 'source_text']])

,qa_id,act_title,section_number,source_text


In [ ]:
import re
import numpy as np

def extract_section_from_text(row):
    if pd.isna(row['section_number']) and pd.notna(row['source_text']):
        match = re.match(r"^\s*(\d+)[\.৷]", row['source_text'])
        if match:
            return match.group(1)
    return row['section_number']

# Apply the function to update the section_number
df['section_number'] = df.apply(extract_section_from_text, axis=1)

# Check how many are still missing after this operation
after_regex_missing = df['section_number'].isna().sum()
print(f"Missing section_number after regex extraction: {after_regex_missing}")

# Display rows that still have a missing section_number
if after_regex_missing > 0:
    print("Rows still missing a section_number after regex extraction:")
    display(df[df['section_number'].isna()][['qa_id', 'act_title', 'section_number', 'source_text']])
else:
    print("All missing section numbers have been filled by regex extraction.")

# Save the updated DataFrame back to CSV
df.to_csv(qa_csv_path, index=False, encoding="utf-8-sig")
print(f"Updated CSV saved to {qa_csv_path}")

Missing section_number after regex extraction: 0
All missing section numbers have been filled by regex extraction.
Updated CSV saved to /content/drive/MyDrive/BDLegalRAG/qa_data/qa_pairs_draft.csv


In [ ]:
import re
import pandas as pd

qa_csv_path = os.path.join(QA_DIR, "qa_pairs_draft.csv")
df = pd.read_csv(qa_csv_path)

def word_overlap_ratio(answer, source):
    """What fraction of the answer's words actually appear in the source text.
    Low overlap = the answer may contain unsupported claims worth double-checking."""
    if not isinstance(answer, str) or not isinstance(source, str):
        return 0.0
    answer_words = set(re.findall(r"[\u0980-\u09FFA-Za-z]+", answer))
    source_words = set(re.findall(r"[\u0980-\u09FFA-Za-z]+", source))
    if not answer_words:
        return 0.0
    return len(answer_words & source_words) / len(answer_words)

OVERLAP_THRESHOLD = 0.5  # নিজের পছন্দমতো tune করতে পারেন — কম মানে বেশি flag হবে

df["overlap_ratio"] = df.apply(
    lambda r: word_overlap_ratio(r["model_draft_answer"], r["source_text"]), axis=1
)
df["needs_careful_review"] = df["overlap_ratio"] < OVERLAP_THRESHOLD

# section_number missing হলেও careful review-এর তালিকায় রাখা ভালো
df["needs_careful_review"] = df["needs_careful_review"] | df["section_number"].isna()

flagged_count = df["needs_careful_review"].sum()
print(f"Total rows: {len(df)}")
print(f"Flagged for careful review: {flagged_count} ({flagged_count/len(df)*100:.0f}%)")
print(f"Can be quickly skimmed: {len(df) - flagged_count}")

# flagged row গুলো উপরে এনে সাজানো, যাতে Sheets/Excel-এ প্রথমেই এগুলো দেখা যায়
df_sorted = df.sort_values("needs_careful_review", ascending=False)
df_sorted.to_csv(qa_csv_path, index=False, encoding="utf-8-sig")
print(f"\nRe-saved with 'overlap_ratio' and 'needs_careful_review' columns, flagged rows sorted to top.")


Total rows: 180
Flagged for careful review: 28 (16%)
Can be quickly skimmed: 152

Re-saved with 'overlap_ratio' and 'needs_careful_review' columns, flagged rows sorted to top.


In [ ]:
print("Summary statistics for 'overlap_ratio':")
display(df['overlap_ratio'].describe())

Summary statistics for 'overlap_ratio':


,overlap_ratio
count,180.000000
mean,0.729076
std,0.176886
min,0.260870
25%,0.617081
50%,0.730303
75%,0.872596
max,1.000000


## ধাপ ৩: Manual Verification — এখন আপনার কাজ

`qa_pairs_draft.csv` ফাইলটা Google Sheets/Excel-এ খুলুন (Drive-এ আছে)। প্রতিটা row-এর জন্য:

1. `source_text` পড়ুন
2. `question` আর `model_draft_answer` টা ঠিক আছে কিনা check করুন
3. ঠিক থাকলে: `human_verified` কলামে **TRUE** লিখুন
4. উত্তর ভুল/অসম্পূর্ণ হলে: `corrected_answer` কলামে সঠিক উত্তর লিখুন, `human_verified` = **TRUE**
5. প্রশ্নটাই অর্থহীন/অপ্রাসঙ্গিক হলে: `human_verified` = **FALSE** (এই row বাদ যাবে)

**সময়ের হিসাব:** প্রতি প্রশ্নে ৩০-৪৫ সেকেন্ড ধরলে ১৮০টা প্রশ্ন verify করতে মোটামুটি ২-৩ ঘণ্টা লাগবে — একদিনে শেষ করা সম্ভব।


In [ ]:
# # Verify করা CSV আবার আপলোড/save করার পর, এই cell চালিয়ে final gold set বানান

# verified_csv_path = os.path.join(QA_DIR, "qa_pairs_draft.csv")  # verify করার পর একই ফাইলে save করবেন
# df = pd.read_csv(verified_csv_path)

# verified_df = df[df["human_verified"].astype(str).str.upper() == "TRUE"].copy()

# # corrected_answer থাকলে সেটাই final answer, নাহলে model-এর draft
# verified_df["final_answer"] = verified_df.apply(
#     lambda row: row["corrected_answer"] if pd.notna(row["corrected_answer"]) and str(row["corrected_answer"]).strip()
#     else row["model_draft_answer"],
#     axis=1
# )

# final_qa_path = os.path.join(QA_DIR, "gold_qa_set_v1.csv")
# final_cols = ["qa_id", "question", "final_answer", "act_title", "act_no", "act_year", "section_number", "source_text"]
# verified_df[final_cols].to_csv(final_qa_path, index=False, encoding="utf-8-sig")

# print(f"Final gold QA set: {len(verified_df)} verified pairs")
# print(f"Saved to: {final_qa_path}")

# Verify করা xlsx আবার আপলোড/save করার পর, এই cell চালিয়ে final gold set বানান

verified_xlsx_path = os.path.join(QA_DIR, "qa_pairs_draft.xlsx")  # verify করার পর একই ফাইলে save করবেন
df = pd.read_excel(verified_xlsx_path)

verified_df = df[df["human_verified"].astype(str).str.upper() == "TRUE"].copy()

# corrected_answer কলাম থাকলে সেটাই final answer, না থাকলে model-এর draft
if "corrected_answer" in verified_df.columns:
    verified_df["final_answer"] = verified_df.apply(
        lambda row: row["corrected_answer"] if pd.notna(row["corrected_answer"]) and str(row["corrected_answer"]).strip()
        else row["model_draft_answer"],
        axis=1
    )
else:
    verified_df["final_answer"] = verified_df["model_draft_answer"]

final_qa_path = os.path.join(QA_DIR, "gold_qa_set_v1.csv")
final_cols = ["qa_id", "question", "final_answer", "act_title", "act_no", "act_year", "section_number", "source_text"]
verified_df[final_cols].to_csv(final_qa_path, index=False, encoding="utf-8-sig")

print(f"Final gold QA set: {len(verified_df)} verified pairs (out of {len(df)} total)")
print(f"Saved to: {final_qa_path}")

Final gold QA set: 180 verified pairs (out of 180 total)
Saved to: /content/drive/MyDrive/BDLegalRAG/qa_data/gold_qa_set_v1.csv


## ধাপ ৪: Sanity Check — Coverage দেখে নিন

Final gold set-টা কতগুলো ভিন্ন Act থেকে এসেছে, দেখে নিন — যদি বেশিরভাগ প্রশ্ন মাত্র কয়েকটা Act থেকে এসে থাকে, sampling-টা আবার balance করা দরকার হতে পারে।


In [ ]:
print("Unique acts covered:", verified_df["act_title"].nunique(), "out of", len(by_act))
print()
print("Questions per act (top 10):")
print(verified_df["act_title"].value_counts().head(10))


Unique acts covered: 49 out of 49

Questions per act (top 10):
act_title
বাংলাদেশ শ্রম আইন, ২০০৬                               29
বাংলাদেশ ইপিজেড শ্রম আইন, ২০১৯                        18
শিশু আইন, ২০১৩                                        11
শিশু দিবাযত্ন কেন্দ্র আইন, ২০২১                        9
স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭          7
পারিবারিক আদালত আইন, ২০২৩                              6
বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭             6
ব্যক্তি মালিকানাধীন সড়ক পরিবহন শ্রমিক কল্যাণ তহবিল     5
সুরক্ষিত লেনদেন (অস্থাবর সম্পত্তি) আইন, ২০২৩           5
ভূমি সংস্কার আইন, ২০২৩                                 5
Name: count, dtype: int64


In [ ]:
import pandas as pd
gold_df = pd.read_csv(os.path.join(QA_DIR, "gold_qa_set_v1.csv"))
print("Total QA pairs:", len(gold_df))
print("Unique acts covered:", gold_df["act_title"].nunique())
print(gold_df["section_number"].isna().sum(), "rows still missing section_number")

Total QA pairs: 180
Unique acts covered: 49
0 rows still missing section_number


## পরের ধাপ

এখন আপনার হাতে `gold_qa_set_v1.csv` — Stage 3 (Chunking Experiment)-এর Precision@k/Recall@k evaluation-এ এটাই ground truth হিসেবে ব্যবহার হবে।

**Supplementary হিসেবে যোগ করতে পারেন (optional, আগে আলোচনা হয়েছিল):** Bangladesh Bar Council exam-এর পুরনো MCQ প্রশ্নগুলো — এটা একটা external, standardized benchmark যোগ করবে, যা শুধু নিজের বানানো QA-এর উপর নির্ভরতা কমাবে এবং Mina paper-এর সাথে সরাসরি তুলনাযোগ্য করবে।
